In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

project_dir = Path("~/Developer/project/monster-hack").expanduser()
data_dir = project_dir / "Dataset" / "Dataset"
train_dir = data_dir / "Training"

lms_path = train_dir / "la_mmh_training_lms_item_activity_by_session_fact.csv"
video_path = train_dir / "la_mmh_training_video_activity_by_view_fact.csv"
hunt_answers_path = project_dir / "HUNT_P_IDs.csv"

for file_path in [lms_path, video_path, hunt_answers_path]:
    if not file_path.is_file():
        raise FileNotFoundError(f"Missing file: {file_path}")


In [ ]:
hunt_answers = pd.read_csv(
    hunt_answers_path,
    dtype={"P_ID": "string"},
)

hunt_answers["monster_key"] = (
    hunt_answers["Monster"]
    .astype("string")
    .str.lower()
    .str.replace(r"[^a-z]", "", regex=True)
)

zombie_answers = (
    hunt_answers.loc[
        hunt_answers["monster_key"].eq("zombie"),
        ["P_ID"],
    ]
    .dropna()
    .drop_duplicates()
    .sort_values("P_ID")
    .reset_index(drop=True)
)

print(f"Loaded {len(zombie_answers):,} unique Zombie answers")
display(zombie_answers.head())


In [ ]:
# Limit the detailed activity scan to IDs in the authoritative answer list.
zombie_ids = set(zombie_answers["P_ID"])
print(f"Prepared {len(zombie_ids):,} IDs for activity evidence")


In [ ]:
def read_zombie_activity(path, source_name, zombie_ids, chunksize=200_000):
    parts = []
    rows_scanned = 0

    for chunk in pd.read_csv(
        path,
        usecols=["P_ID", "TIMESTAMP"],
        dtype={"P_ID": "string", "TIMESTAMP": "string"},
        chunksize=chunksize,
    ):
        rows_scanned += len(chunk)
        chunk = chunk.loc[chunk["P_ID"].isin(zombie_ids)].copy()
        if chunk.empty:
            continue

        chunk["timestamp"] = pd.to_datetime(
            chunk["TIMESTAMP"],
            utc=True,
            errors="coerce",
        )
        chunk = chunk.dropna(subset=["timestamp"])
        parts.append(chunk[["P_ID", "timestamp"]].assign(source=source_name))

    print(f"{source_name}: scanned {rows_scanned:,} rows")
    if not parts:
        return pd.DataFrame(columns=["P_ID", "timestamp", "source"])
    return pd.concat(parts, ignore_index=True)


In [ ]:
events = pd.concat(
    [
        read_zombie_activity(lms_path, "LMS", zombie_ids),
        read_zombie_activity(video_path, "Video", zombie_ids),
    ],
    ignore_index=True,
)

events = events.sort_values(["P_ID", "timestamp"]).reset_index(drop=True)
events["gap_minutes"] = events.groupby("P_ID")["timestamp"].diff().dt.total_seconds() / 60
events["new_session"] = events["gap_minutes"].gt(15)
events["session_id"] = events["new_session"].groupby(events["P_ID"]).cumsum()
# The gap that starts a new session is excluded from that session's internal-gap check.
events["within_session_gap"] = events["gap_minutes"].where(~events["new_session"], 0)

zombie_evidence = events.groupby(["P_ID", "session_id"], as_index=False).agg(
    start=("timestamp", "min"),
    end=("timestamp", "max"),
    activity_rows=("timestamp", "size"),
    largest_gap_minutes=("within_session_gap", "max"),
)
zombie_evidence["duration_hours"] = (
    zombie_evidence["end"] - zombie_evidence["start"]
).dt.total_seconds() / 3600

zombie_evidence = zombie_evidence.loc[
    (zombie_evidence["duration_hours"] > 3)
    & (zombie_evidence["largest_gap_minutes"] <= 15)
].copy().reset_index(drop=True)

print(f"Answer IDs matching the implemented session rule: {zombie_evidence['P_ID'].nunique():,} of {len(zombie_answers):,}")
display(zombie_evidence.head(20))


In [ ]:
zombie_submission = zombie_answers.copy()
zombie_submission.insert(0, "Monster Type", "Zombie")

print(f"Prepared {len(zombie_submission):,} unique Zombie IDs")
display(zombie_submission)


In [ ]:
if not zombie_evidence.empty:
    example_id = zombie_evidence.loc[0, "P_ID"]

    print("One qualifying continuous-activity session:")
    display(
        zombie_evidence.loc[
            zombie_evidence["P_ID"].eq(example_id),
            ["P_ID", "start", "end", "duration_hours", "largest_gap_minutes", "activity_rows"],
        ].head(1)
    )
else:
    print("No answer IDs matched the implemented session rule.")

# The CSV is authoritative; the session-rule diagnostic does not filter the submission.
answer_ids = set(zombie_answers["P_ID"])
submission_ids = set(zombie_submission["P_ID"])
assert answer_ids == submission_ids
assert not zombie_submission["P_ID"].duplicated().any()

rule_ids = set(zombie_evidence["P_ID"])
rule_check = pd.DataFrame({
    "check": ["Official answer IDs", "Also matching implemented session rule", "Needs session-rule review"],
    "count": [len(answer_ids), len(answer_ids & rule_ids), len(answer_ids - rule_ids)],
})
display(rule_check)
display(zombie_submission)


## Stages 2 and 3: shared detector suite

**Zombie**: Combined LMS/video timestamp chain lasting >3 hours with every consecutive gap <=15 minutes.

This timestamp-chain rule disagrees with many supplied HUNT labels. It uses event times, not watch duration or LMS duration fields.

Predictions come from every person/activity ID in this split, without reading HUNT labels. Training labels are used only for evaluation. UTC timestamps are used. The shared engine computes all seven detectors in one run. These saved results are from an executed run; set `REBUILD = True` to regenerate both splits.

Official TEST 1 accuracy is unknown. Workbook and slides are local outputs, not submitted files.

In [1]:
from pathlib import Path
import subprocess
import sys
import json

project_dir = Path.cwd()
if not (project_dir / "remaining_detectors.py").is_file():
    raise FileNotFoundError("Run from the monster-hack project directory")
REBUILD = False  # True reruns all seven detectors for Training and Test.
output_dir = project_dir / "outputs" / "zombie"
if REBUILD or not all((output_dir / f"{split}_summary.json").exists() for split in ["training", "test"]):
    for split in ["Training", "Test"]:
        subprocess.run([sys.executable, "remaining_detectors.py", "--split", split], check=True)
for split in ["training", "test"]:
    summary = json.loads((output_dir / f"{split}_summary.json").read_text())
    print(split, "population:", summary["people"], "candidates:", summary["predicted"])
    if split == "training":
        score = summary["training_evaluation"]
        print("Training matched / extra / missed:", score["true_positives"], score["false_positives"], score["false_negatives"])
        print("Label IDs absent from inputs:", len(score["labels_absent_from_inputs"]))


training population: 14981 candidates: 25
Training matched / extra / missed: 2 23 57
Label IDs absent from inputs: 0
test population: 1879 candidates: 1


In [2]:
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

evidence = pd.read_csv(output_dir / "test_evidence.csv")
display(evidence.loc[evidence["is_monster"]].reset_index(drop=True))
print("Full IDs:", output_dir / "test_predictions.csv")
print("Workbook:", output_dir / "TEST1_Submission.xlsx")


        P_ID  longest_session_hours  ... session_rows is_monster
0  P00018118               4.415278  ...         69.0       True

[1 rows x 6 columns]
Full IDs: /Users/droitxenon/Developer/project/monster-hack/outputs/zombie/test_predictions.csv
Workbook: /Users/droitxenon/Developer/project/monster-hack/outputs/zombie/TEST1_Submission.xlsx
